In [5]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def get_data_from_kaggle():
    from kagglehub.datasets import dataset_download
    downloaded_path = dataset_download(
        "blastchar/telco-customer-churn",
        path="WA_Fn-UseC_-Telco-Customer-Churn.csv",
        output_dir=".",
    )
    os.rename(downloaded_path, "./telco.csv")

In [6]:
# ==================== 0. Check if the dataset exists, if not download it from Kaggle ====================
if not os.path.exists(r"./telco.csv"):
    print("download")
    get_data_from_kaggle()

assert os.path.exists(r"./telco.csv"), "Dataset not found. Please download it from Kaggle."

In [7]:
# ==================== 1. Load and inspect the data ====================
# 3.2
data = pd.read_csv(r"./telco.csv")
data = data.convert_dtypes()

data.head()
data.info()
display(data.describe())
print("shape:", data.shape)

# 3.8 & 3.4
display(data.describe())
print("Num of negative monthly charge: ", data[data["MonthlyCharges"] < 0].shape[0])

# 3.6
# ---------- 1.1 change `No internet service` to `No`.
data.replace({ "No internet service": "No", "No phone service": "No"}, inplace=True)

# ---------- 1.2 Convert Yes/No and Male/Female to boolean(integer).
# 3.9 & 3.17
for col in data.columns:
    st = set(data[col].unique())
    if st == {"Yes", "No"}: data[col] = data[col].map({"Yes": 1, "No": 0})
    elif st == {"Male", "Female"}: data[col] = data[col].map({"Female": 0, "Male": 1})

# ---------- 1.3 Drop the unnessary data.
# 3.11
data = data.drop(columns=["customerID", "gender"])
data.info()

# 3.10
print(data.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   string 
 1   gender            7043 non-null   string 
 2   SeniorCitizen     7043 non-null   Int64  
 3   Partner           7043 non-null   string 
 4   Dependents        7043 non-null   string 
 5   tenure            7043 non-null   Int64  
 6   PhoneService      7043 non-null   string 
 7   MultipleLines     7043 non-null   string 
 8   InternetService   7043 non-null   string 
 9   OnlineSecurity    7043 non-null   string 
 10  OnlineBackup      7043 non-null   string 
 11  DeviceProtection  7043 non-null   string 
 12  TechSupport       7043 non-null   string 
 13  StreamingTV       7043 non-null   string 
 14  StreamingMovies   7043 non-null   string 
 15  Contract          7043 non-null   string 
 16  PaperlessBilling  7043 non-null   string 
 17  Paymen

,SeniorCitizen,tenure,MonthlyCharges
count,7043.0,7043.0,7043.0
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.0,0.0,18.25
25%,0.0,9.0,35.5
50%,0.0,29.0,70.35
75%,0.0,55.0,89.85
max,1.0,72.0,118.75


shape: (7043, 21)


,SeniorCitizen,tenure,MonthlyCharges
count,7043.0,7043.0,7043.0
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.0,0.0,18.25
25%,0.0,9.0,35.5
50%,0.0,29.0,70.35
75%,0.0,55.0,89.85
max,1.0,72.0,118.75


Num of negative monthly charge:  0
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SeniorCitizen     7043 non-null   Int64  
 1   Partner           7043 non-null   int64  
 2   Dependents        7043 non-null   int64  
 3   tenure            7043 non-null   Int64  
 4   PhoneService      7043 non-null   int64  
 5   MultipleLines     7043 non-null   int64  
 6   InternetService   7043 non-null   string 
 7   OnlineSecurity    7043 non-null   int64  
 8   OnlineBackup      7043 non-null   int64  
 9   DeviceProtection  7043 non-null   int64  
 10  TechSupport       7043 non-null   int64  
 11  StreamingTV       7043 non-null   int64  
 12  StreamingMovies   7043 non-null   int64  
 13  Contract          7043 non-null   string 
 14  PaperlessBilling  7043 non-null   int64  
 15  PaymentMethod     7043 non-null   string 
 16  MonthlyCharges    

In [8]:
# ==================== 2. Define the variables ====================
# ---------- 2.1 One-hot encode the categorical variables.
data = pd.get_dummies(data, columns=["InternetService"], drop_first=True)
bool_cols = data.select_dtypes(include=["boolean"]).columns
data[bool_cols] = data[bool_cols].astype("Int64")

print(data.head(1))

# ---------- 2.2 Define the features and target variable.
# 3.7
COLUMN_MAP={
    "PhoneService": "phone_service", "MultipleLines": "multiple_lines", 
    "OnlineSecurity": "online_security", "OnlineBackup": "online_backup", 
    "DeviceProtection": "device_protection", "TechSupport": "tech_support", 
    "StreamingTV": "streaming_tv", "StreamingMovies": "streaming_movies", 
    "InternetService_Fiber optic": "internetService_fiber_optic", 
    "InternetService_No": "internetservice_no",
}
data.rename(columns=COLUMN_MAP, inplace=True)
print(data.columns)

print(data.head(1))

service_cols = [
    "phone_service", "multiple_lines", "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies", "internetService_fiber_optic", 
    "internetservice_no"
]

features = data[service_cols]
target = data["MonthlyCharges"]

# features.info()
# target.info()

   SeniorCitizen  Partner  Dependents  tenure  PhoneService  MultipleLines  \
0              0        1           0       1             0              0   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  StreamingTV  \
0               0             1                 0            0            0   

   StreamingMovies        Contract  PaperlessBilling     PaymentMethod  \
0                0  Month-to-month                 1  Electronic check   

   MonthlyCharges TotalCharges  Churn  InternetService_Fiber optic  \
0           29.85        29.85      0                            0   

   InternetService_No  
0                   0  
Index(['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'phone_service',
       'multiple_lines', 'online_security', 'online_backup',
       'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'Churn', 'internetService_fiber_

In [9]:
# ==================== 3. Split the data ==========================================
X_train, X_test, y_train, y_test = train_test_split(features, target, train_size=0.8, random_state=818)


# ==================== 4. Fit a multivariable linear regression ====================
model = LinearRegression()
model.fit(X_train, y_train)


# ==================== 5. Report the parameters ====================
# print("Coefficients:", model.coef_)
# print("Intercept:", model.intercept_)

b = model.intercept_  # base plan charge
# 3.1, 3.5
coefficient_list = (pd.DataFrame({'service': service_cols, 'cost': model.coef_.round(2)})
                    .sort_values('cost', ascending=False)
                    .reset_index(drop=True))  # dollar contribution of each service

print(f"Base monthly charge (intercept): ${b:.2f}")
print(coefficient_list)

Base monthly charge (intercept): $24.95
                       service   cost
0  internetService_fiber_optic  24.93
1                phone_service  20.08
2             streaming_movies   9.99
3                 streaming_tv   9.95
4                 tech_support   5.06
5              online_security   5.02
6               multiple_lines   5.01
7            device_protection   5.01
8                online_backup   4.97
9           internetservice_no -25.07


In [10]:
# ==================== 6. Evaluate on the test set ====================
mae = mean_absolute_error(y_test, model.predict(X_test))
mse = mean_squared_error(y_test, model.predict(X_test))
rmse = np.sqrt(mse)
r2 = r2_score(y_test, model.predict(X_test))
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(f"R-squared: {r2:.2f}")

Mean Absolute Error: 0.80
Root Mean Squared Error: 1.06
R-squared: 1.00


In [11]:
# ==================== 7. Compare to a baseline ====================
# ---------- 7.1 Data preparation
addon_cols = [
    "phone_service", "multiple_lines", "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies", "internetService_fiber_optic", 
    "internetservice_no"
]

# 3.1 & 3.8
data_new = pd.DataFrame({
    "num_addons": data[addon_cols].sum(axis=1),  # sum across the row = how many add-ons
    "MonthlyCharges": data["MonthlyCharges"],
})

# ---------- 7.2 Split the data and train a baseline model
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(data_new[["num_addons"]], data_new["MonthlyCharges"], train_size=0.8, random_state=818)
base_model = LinearRegression()
base_model.fit(X_train_new, y_train_new)

# ---------- 7.3 Evaluate the baseline model
mae_base = mean_absolute_error(y_test_new, base_model.predict(X_test_new))
mse_base = mean_squared_error(y_test_new, base_model.predict(X_test_new))
rmse_base = np.sqrt(mse_base)
r2_base = r2_score(y_test_new, base_model.predict(X_test_new))
print(f"Baseline Mean Absolute Error: {mae_base:.2f}")
print(f"Baseline Root Mean Squared Error: {rmse_base:.2f}")
print(f"Baseline R-squared: {r2_base:.2f}")

Baseline Mean Absolute Error: 14.19
Baseline Root Mean Squared Error: 16.44
Baseline R-squared: 0.72


In [12]:
# ==================== 8. Interpret the results ====================
# 3.5
coef_table = pd.Series(model.coef_, index=service_cols).sort_values(ascending=False)
print(f"Base monthly charge (intercept): ${model.intercept_:.2f}")
print("Estimated $/month contribution of each service:")
for name, coef in coef_table.items(): print(f"  {name}: {coef:+.2f} $/month")

# higher R2 is better. (Multi Variable Model R2 is higher, so it's better)
# lower MAE & RMSE is better. (Multi Variable Model MAE and RMSE is lower, so it's better)

Base monthly charge (intercept): $24.95
Estimated $/month contribution of each service:
  internetService_fiber_optic: +24.93 $/month
  phone_service: +20.08 $/month
  streaming_movies: +9.99 $/month
  streaming_tv: +9.95 $/month
  tech_support: +5.06 $/month
  online_security: +5.02 $/month
  multiple_lines: +5.01 $/month
  device_protection: +5.01 $/month
  online_backup: +4.97 $/month
  internetservice_no: -25.07 $/month


In [ ]:
# ==================== 9. Predict a new customer ====================
# 3.1
new_customer = pd.DataFrame([{
    "phone_service": 1,
    "multiple_lines": 1,
    "online_security": 1,
    "online_backup": 0,
    "device_protection": 0,
    "tech_support": 1,
    "streaming_tv": 1,
    "streaming_movies": 0,
    "internetService_fiber_optic": 1,
    "internetservice_no": 0,
}], columns=service_cols)

predicted_charge = model.predict(new_customer)[0]
print(f"Predicted monthly charge for the new customer: ${predicted_charge:.2f}")

# Print cleaned data
data.to_csv(r"cleaned_data.csv", index=False)

phone_service                  0
multiple_lines                 0
online_security                0
online_backup                  0
device_protection              0
tech_support                   0
streaming_tv                   0
streaming_movies               0
internetService_fiber_optic    0
internetservice_no             0
dtype: int64
Predicted monthly charge for the new customer: $94.99
